# Customer Churn Prediction

An end-to-end machine learning project to predict whether a telecom customer is likely to churn.

### Project Workflow

1. Business Problem
2. Dataset Overview
3. Data Understanding
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Feature Preparation
7. Train-Test Split
8. Data Preprocessing
9. Model Building
10. Model Evaluation
11. Model Comparison
12. Cross-Validation
13. Final Model Evaluation
14. Prediction on a New Customer
15. Save the Model
16. Conclusion

## 1. Business Problem

### Business Problem

Customer churn occurs when a customer stops using a company's products or services.

The company wants to identify customers who may be at higher risk of churn so that appropriate customer-retention strategies can potentially be considered.

### Machine Learning Objective

Build a **binary classification model** that predicts whether a customer will churn based on demographic, service, contract, and billing information.

### Target Variable

**Churn**

- `Yes` → Customer churns
- `No` → Customer does not churn

### Project Goal

Develop and compare multiple classification models and evaluate them using metrics that are appropriate for an imbalanced churn-prediction problem.

## 2. Dataset Overview

The dataset contains customer-level information about telecom services, demographics, contracts, and billing.

Each row represents one customer, while each column contains a customer attribute or the target churn status.

In [ ]:
# Import Pandas for data manipulation and analysis
import pandas as pd

# Load the telecom customer churn dataset.
# The dataset is stored inside a ZIP archive containing the CSV file.
df = pd.read_csv("C:/Users/Vaishnavi/Downloads/archive (7).zip")

# Display the first five rows to inspect the dataset structure.
df.head()

In [ ]:
# Check the number of rows and columns in the dataset.
df.shape

### Dataset Shape

The dataset contains **7,043 rows and 21 columns**.

The original columns include customer demographics, services, contract information, billing information, and the `Churn` target.

## 3. Data Understanding

In [ ]:
# Examine column names, data types, and non-null values.
# This helps identify numerical and categorical features
# and detect potential data-quality issues.
df.info()

In [ ]:
# Generate descriptive statistics for numerical features.
# This helps understand the range and distribution of numerical variables.
df.describe()

In [ ]:
# Check the number of missing values in each column.
df.isnull().sum()

In [ ]:
# Inspect the first few unique values of TotalCharges.
# This helps identify unexpected entries such as blank strings.
df["TotalCharges"].unique()[:20]

### Initial Data Quality Check

The initial inspection is used to identify:

- Missing values
- Incorrect or inconsistent data types
- Numerical and categorical variables
- Potential data-quality issues
- Duplicate records

## 4. Data Cleaning

### Handling `TotalCharges`

The `TotalCharges` column contains blank-space values instead of standard missing values.

These values need to be converted into proper missing values before the column can be converted to a numerical data type.

In [ ]:
# Identify rows where TotalCharges contains blank spaces.
df[df["TotalCharges"].str.strip() == ""]

In [ ]:
# Count the number of blank-space values in TotalCharges.
df["TotalCharges"].str.strip().eq("").sum()

In [ ]:
# Convert blank spaces into proper missing values.
df["TotalCharges"] = df["TotalCharges"].replace(" ", pd.NA)

In [ ]:
# Convert TotalCharges from string/object format to numerical format.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

In [ ]:
# Check how many missing values remain after conversion.
df["TotalCharges"].isna().sum()

In [ ]:
# Replace missing TotalCharges values with 0.
# These records correspond to customers with zero tenure,
# for whom no accumulated charges have been recorded.
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
# Verify that TotalCharges no longer contains missing values.
df["TotalCharges"].isna().sum()

In [ ]:
# Confirm the updated data types after cleaning.
df.info()

### Duplicate Records

Duplicate rows can cause certain customers to receive unintended additional weight during model training.

In [ ]:
# Check for duplicate customer records.
df.duplicated().sum()

No duplicate rows were found in the dataset.

### Target Variable Distribution

In [ ]:
# Count the number of customers in each churn category.
df["Churn"].value_counts()

In [ ]:
# Calculate the percentage distribution of the target classes.
df["Churn"].value_counts(normalize=True) * 100

### Observation

The target variable is moderately imbalanced:

- **No Churn:** 5,174 customers (73.46%)
- **Churn:** 1,869 customers (26.54%)

Because the classes are not evenly distributed, model evaluation should not rely on accuracy alone. Precision, recall, F1-score, and ROC-AUC are also considered.

### Categorical Feature Inspection

In [ ]:
# Display the unique categories present in each categorical column.
# This helps understand the possible values before preprocessing.
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())

## 5. Exploratory Data Analysis (EDA)

EDA is used to understand patterns and relationships in the data before model training.

The analysis below focuses on variables that may be useful for understanding customer churn.

In [ ]:
# Import visualization libraries.
import matplotlib.pyplot as plt
import seaborn as sns

### 5.1 Customer Churn Distribution

In [ ]:
# Visualize the distribution of customers who churned
# versus customers who did not churn.
df["Churn"].value_counts().plot(kind="bar")

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

### Observation

The dataset contains substantially more customers who did not churn than customers who churned.

### 5.2 Contract Type vs Churn

In [ ]:
# Calculate the percentage of churn and non-churn customers
# within each contract type.
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

contract_churn

### Observation

The observed churn rate differs considerably across contract types.

Month-to-month customers show a substantially higher observed churn rate than customers with one-year or two-year contracts.

This represents an **association in the dataset** and does not by itself establish that contract type causes churn.

### 5.3 Tenure vs Churn

In [ ]:
# Calculate the average customer tenure for each churn category.
df.groupby("Churn")["tenure"].mean()

In [ ]:
# Visualize the distribution of customer tenure across churn categories.
sns.boxplot(
    x="Churn",
    y="tenure",
    data=df
)

plt.title("Tenure Distribution by Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure (Months)")
plt.show()

### Observation

Customers who churned had a lower average tenure than customers who did not churn.

The boxplot also shows that the tenure distributions differ between the two groups.

This is an observed relationship and does not imply causation.

### 5.4 Monthly Charges vs Churn

In [ ]:
# Compare the average monthly charges between churned
# and non-churned customers.
df.groupby("Churn")["MonthlyCharges"].mean()

In [ ]:
# Visualize the distribution of monthly charges across churn categories.
sns.boxplot(
    x="Churn",
    y="MonthlyCharges",
    data=df
)

plt.title("Monthly Charges Distribution by Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

### Observation

Customers who churned had higher average monthly charges than customers who did not churn.

The distributions also show a shift toward higher monthly charges among churned customers.

### 5.5 Internet Service vs Churn

In [ ]:
# Calculate the percentage of churn and non-churn customers
# within each InternetService category.
internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

internet_churn

### Observation

The observed churn rate varies across internet service categories.

Fiber optic customers have a higher observed churn rate than the other internet service categories in this dataset.

This is an association and does not establish a causal relationship.

### 5.6 Total Charges vs Churn

In [ ]:
# Compare the average total charges between churned
# and non-churned customers.
df.groupby("Churn")["TotalCharges"].mean()

### Observation

Customers who churned had lower average total charges.

This is consistent with the lower average tenure observed among churned customers, since customers who stay longer generally accumulate more total charges.

## 6. Feature Preparation

### Removing the Customer Identifier

`customerID` is a unique identifier rather than a meaningful predictive feature. It is therefore removed before model training.

In [ ]:
# Remove customerID because it is a unique identifier
# and does not provide meaningful predictive information.
df = df.drop("customerID", axis=1)

In [ ]:
# Separate input features from the target variable.
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Check the feature matrix dimensions.
print("Feature matrix shape:", X.shape)

# Check the target variable dimensions.
print("Target shape:", y.shape)

After removing `customerID`, the model uses **19 input features** to predict the `Churn` target.

## 7. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets.
# 80% of the data is used for training and 20% for testing.
# Stratification preserves the churn class distribution
# in both datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Verify the dimensions of the training and testing datasets.
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Check the churn distribution in the training set.
y_train.value_counts(normalize=True)

In [ ]:
# Check the churn distribution in the testing set.
y_test.value_counts(normalize=True)

### Why Stratification?

Stratification ensures that the proportion of churn and non-churn customers remains approximately the same in both the training and testing datasets.

## 8. Data Preprocessing

In [ ]:
# Define numerical features that require scaling.
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [ ]:
# Identify categorical features automatically by selecting
# columns that are not part of the numerical feature list.
categorical_features = [
    col for col in X.columns
    if col not in numerical_features
]

# Display the categorical features.
categorical_features

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Standardize numerical features and one-hot encode categorical features.
#
# StandardScaler puts numerical variables on a comparable scale.
# OneHotEncoder converts categorical variables into numerical columns.
# handle_unknown="ignore" allows the model to handle unseen categories
# when making predictions on new data.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

### Target Encoding

In [ ]:
# Convert target labels into binary numerical values:
# No Churn = 0
# Churn = 1

y_train = y_train.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})

print(y_train.head())
print(y_test.head())

## 9. Model Building

### 9.1 Logistic Regression

Logistic Regression is used as a baseline binary classification model.

`class_weight="balanced"` gives additional importance to the minority churn class, which is useful because the target variable is moderately imbalanced.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Create a complete pipeline containing preprocessing and Logistic Regression.
# Keeping preprocessing inside the pipeline helps prevent data leakage
# and ensures the same transformations are applied during prediction.

logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
# Train the Logistic Regression pipeline on the training data.
logreg_pipeline.fit(X_train, y_train)

In [ ]:
# Generate class predictions for the test set.
y_pred_logreg = logreg_pipeline.predict(X_test)

In [ ]:
# Generate predicted probabilities for the positive class (Churn = 1).
# These probabilities are required for ROC-AUC evaluation.
y_prob_logreg = logreg_pipeline.predict_proba(X_test)[:, 1]

### 9.2 Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Import evaluation metrics used to compare the classification models.

In [ ]:
# Calculate evaluation metrics for Logistic Regression.
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
precision_logreg = precision_score(y_test, y_pred_logreg)
recall_logreg = recall_score(y_test, y_pred_logreg)
f1_logreg = f1_score(y_test, y_pred_logreg)
roc_auc_logreg = roc_auc_score(y_test, y_prob_logreg)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {accuracy_logreg:.4f}")
print(f"Precision: {precision_logreg:.4f}")
print(f"Recall   : {recall_logreg:.4f}")
print(f"F1 Score : {f1_logreg:.4f}")
print(f"ROC-AUC  : {roc_auc_logreg:.4f}")

**Metric interpretation**

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Of the customers predicted to churn, how many actually churned?
- **Recall:** Of all customers who actually churned, how many were correctly identified?
- **F1 Score:** Balance between precision and recall.
- **ROC-AUC:** Measures how well the model separates churn from non-churn across classification thresholds.

Because the objective includes identifying potential churners, recall and ROC-AUC are particularly important alongside the other metrics.

### 9.3 Logistic Regression Confusion Matrix

In [ ]:
# Create a confusion matrix to examine correct and incorrect predictions.
cm_logreg = confusion_matrix(y_test, y_pred_logreg)

print(cm_logreg)

In [ ]:
# Visualize the confusion matrix.
plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_logreg,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

For the Logistic Regression model, the confusion matrix contains:

- **747 True Negatives:** correctly predicted No Churn
- **288 False Positives:** predicted Churn but actually No Churn
- **81 False Negatives:** actual Churn but predicted No Churn
- **293 True Positives:** correctly predicted Churn

### 9.4 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Create a Decision Tree pipeline.
# max_depth limits tree complexity to reduce overfitting.
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
# Train the Decision Tree pipeline.
dt_pipeline.fit(X_train, y_train)

In [ ]:
# Generate class predictions and churn probabilities for the test set.
y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Calculate Decision Tree evaluation metrics.
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

# ROC-AUC must use predicted probabilities rather than hard 0/1 predictions.
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Results")
print("---------------------")
print(f"Accuracy : {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall   : {recall_dt:.4f}")
print(f"F1 Score : {f1_dt:.4f}")
print(f"ROC-AUC  : {roc_auc_dt:.4f}")

### 9.5 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest pipeline.
# n_estimators=200 creates an ensemble of 200 decision trees.
# class_weight="balanced" gives additional importance to the minority class.
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# Train the Random Forest pipeline.
rf_pipeline.fit(X_train, y_train)

In [ ]:
# Generate class predictions and churn probabilities for the test set.
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Calculate Random Forest evaluation metrics.
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

# ROC-AUC must use predicted probabilities rather than hard 0/1 predictions.
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_rf:.4f}")

## 10. Model Comparison

In [ ]:
# Compare all three trained models using multiple evaluation metrics.
# This prevents model selection based on accuracy alone.

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_logreg,
        accuracy_dt,
        accuracy_rf
    ],
    "Precision": [
        precision_logreg,
        precision_dt,
        precision_rf
    ],
    "Recall": [
        recall_logreg,
        recall_dt,
        recall_rf
    ],
    "F1 Score": [
        f1_logreg,
        f1_dt,
        f1_rf
    ],
    "ROC-AUC": [
        roc_auc_logreg,
        roc_auc_dt,
        roc_auc_rf
    ]
})

results.round(4)

### Model Selection Considerations

Accuracy alone can be misleading for a churn problem because the target classes are imbalanced.

The models are therefore compared using accuracy, precision, recall, F1-score, and ROC-AUC. Churn recall is particularly useful when the goal is to identify as many actual churners as possible, while ROC-AUC evaluates overall class-separation ability.

## 11. Cross-Validation

### 5-Fold Stratified Cross-Validation

Five-fold stratified cross-validation is used to evaluate the consistency of ROC-AUC on different subsets of the training data.

The test set remains separate and is used later for the final held-out evaluation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Create a stratified 5-fold cross-validation strategy.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate Logistic Regression using 5-fold cross-validation.
logreg_cv = cross_val_score(
    logreg_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

# Evaluate Decision Tree using the same cross-validation strategy.
dt_cv = cross_val_score(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

# Evaluate Random Forest using the same cross-validation strategy.
rf_cv = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("5-Fold Cross-Validation ROC-AUC")
print("--------------------------------")
print(f"Logistic Regression: {logreg_cv.mean():.4f} ± {logreg_cv.std():.4f}")
print(f"Decision Tree      : {dt_cv.mean():.4f} ± {dt_cv.std():.4f}")
print(f"Random Forest      : {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

### Cross-Validation Result

The Logistic Regression model achieved a mean cross-validation ROC-AUC of approximately **0.8460 ± 0.0124** in the current run.

This was higher than the corresponding mean ROC-AUC values obtained by the Decision Tree and Random Forest models.

Based on the model comparison and cross-validation results, Logistic Regression is selected as the final model for this project.

## 12. Final Model Evaluation

The selected final model is the trained Logistic Regression pipeline.

The model is evaluated once more on the held-out test set to report its final performance.

In [ ]:
# Display the detailed classification report for the final model.
print("FINAL MODEL - LOGISTIC REGRESSION")
print("=" * 40)

print(classification_report(
    y_test,
    y_pred_logreg,
    target_names=["No Churn", "Churn"]
))

In [ ]:
# Create the final confusion matrix.
cm_final = confusion_matrix(y_test, y_pred_logreg)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_final,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Model - Confusion Matrix")
plt.show()

### Final Test-Set Results

The Logistic Regression model achieved:

- **Accuracy:** 73.81%
- **Precision:** 50.43%
- **Recall:** 78.34%
- **F1 Score:** 61.36%
- **ROC-AUC:** 84.16%

The confusion matrix shows that the model correctly identified **293 of the 374 actual churn customers** in the test set, while **81 actual churn customers were missed**.

The model therefore prioritizes identifying potential churners, with some false positives as a trade-off.

## 13. Prediction on a New Customer

The complete trained pipeline is used to predict churn for an illustrative new customer.

Because preprocessing is included inside the pipeline, raw customer information can be passed directly to the model without manually applying scaling or one-hot encoding.

In [ ]:
# Create an illustrative new customer record.
# The feature names and category values match the training data.

new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 4,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.0,
    "TotalCharges": 340.0
}])

new_customer

In [ ]:
# Generate the predicted class.
prediction = logreg_pipeline.predict(new_customer)[0]

# Generate the predicted probability of churn.
churn_probability = logreg_pipeline.predict_proba(new_customer)[0][1]

print(
    "Prediction:",
    "Churn" if prediction == 1 else "No Churn"
)

print(f"Churn Probability: {churn_probability:.2%}")

### Example Prediction

For the illustrative customer used in this example, the model predicted **Churn** with an estimated churn probability of **90.89%** in the current run.

This probability is the model's estimate and does not guarantee that the customer will actually churn.

## 14. Save the Model

The complete preprocessing and Logistic Regression pipeline is saved as a single file.

Saving the entire pipeline ensures that future predictions use the same preprocessing steps as the trained model.

In [ ]:
import joblib

# Save the complete preprocessing + model pipeline.
joblib.dump(
    logreg_pipeline,
    "customer_churn_model.pkl"
)

print("Model saved successfully!")

In [ ]:
# Verify that the model file was created successfully.
import os

print("Model file exists:", os.path.exists("customer_churn_model.pkl"))

## 15. Conclusion

This project developed an end-to-end machine learning pipeline for predicting customer churn.

### Key Steps Completed

- Loaded and inspected 7,043 telecom customer records
- Cleaned missing and inconsistent `TotalCharges` values
- Checked for duplicate records
- Analyzed the target class distribution
- Performed exploratory data analysis
- Removed the non-predictive `customerID` identifier
- Prepared numerical and categorical features
- Applied StandardScaler and OneHotEncoder through a ColumnTransformer
- Used a stratified train-test split
- Trained Logistic Regression, Decision Tree, and Random Forest models
- Evaluated models using accuracy, precision, recall, F1-score, and ROC-AUC
- Performed 5-fold stratified cross-validation
- Selected Logistic Regression based on the observed model comparison and cross-validation results
- Generated an example churn prediction
- Saved the complete trained pipeline using Joblib

### Final Model

The selected Logistic Regression pipeline achieved a test ROC-AUC of **0.8416** and a churn recall of **0.7834** in the current run.

The project demonstrates a complete machine learning workflow from raw customer data to a reusable prediction pipeline.